<a href="https://colab.research.google.com/github/dyjdlopez/mapua_ieemg_decision_science/blob/main/IE151P-S/Module%2001/01_02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01-02: Data Cleaning with Pandas

**IE151P-S — Advanced Computer Applications**

Module 1 · Notebook 01-02


**Companion to:** Lesson 1.3 — *Data Types, Cleaning & Transformation*. Notebook 01-01 built the pandas foundations; this notebook puts every cleaning technique from that lesson into working code, on a real, messy dataset.

---

### Learning Objectives

By the end of this notebook, you will be able to:
- Detect and fix duplicate records, missing values, and sentinel values disguised as real data
- Apply conditional mapping to standardize inconsistent category labels
- Use fuzzy string matching to catch and correct typos in categorical data
- Package cleaning steps into reusable functions and chain them into a single pipeline
- Export a cleaned dataset for use in later work


## 1. Recall: Loading and Selecting Data

Quick refresher from Notebook 01-01: [`pd.read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) loads a table from a URL, and [`.loc[]`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) selects rows and columns by label.

**About this dataset.** We're using the **Adult / Census Income** dataset — real 1994 U.S. Census Bureau data, extracted by Ronny Kohavi and Barry Becker, originally built for predicting whether a person's income exceeds \$50,000/year. It has 32,561 rows and a genuine mix of numeric (`age`, `hours.per.week`, `capital.gain`, `capital.loss`) and categorical (`workclass`, `education`, `marital.status`, `occupation`, `race`, `sex`, `native.country`) columns — and, unlike the Palmer Penguins data, it has real duplicate rows and real sentinel values, which is exactly what this notebook needs.

**Citation:** Kohavi, R. (1996). *Scaling Up the Accuracy of Naive-Bayes Classifiers: a Decision-Tree Hybrid*. Proceedings of the Second International Conference on Knowledge Discovery and Data Mining. Data source: UCI Machine Learning Repository, "Adult" / "Census Income" dataset.


In [1]:
import pandas as pd

adult_url = "https://raw.githubusercontent.com/pooja2512/Adult-Census-Income/master/adult.csv"
adult_df = pd.read_csv(adult_url)

adult_df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [2]:
print("Shape (rows, columns):", adult_df.shape)

Shape (rows, columns): (32561, 15)


A quick recall of row/column selection with `.loc[]` — age, education, and income for the first five rows:


In [3]:
adult_df.loc[0:4, ['age', 'education', 'income']]

,age,education,income
0,90,HS-grad,<=50K
1,82,HS-grad,<=50K
2,66,Some-college,<=50K
3,54,7th-8th,<=50K
4,41,Some-college,<=50K


## 2. Data Type & Null Checks

Before touching anything else, check what pandas thinks each column *is*, and whether it thinks anything is missing.


In [ ]:
adult_df.dtypes

age               int64
workclass           str
fnlwgt            int64
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
sex                 str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
income              str
dtype: object

In [ ]:
adult_df.isna().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

Every column reports **zero** missing values. Given this is real census data with a documented "unknown" pattern, that should feel suspicious — exactly the same suspicion that should hit you when `.describe()` looked *too* clean in Notebook 01-01.

Let's actually look inside a column that's likely to have gaps, using [`.value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html):


In [ ]:
adult_df['workclass'].value_counts()

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

There it is: **`?`**, appearing 1,836 times. This is a **sentinel value** — the exact same pattern as the BP = 999 case from the Lesson 1.3 slides, just spelled differently. pandas has no way to know `"?"` means "missing" unless we tell it, which is exactly why `.isna()` missed it completely.

Let's convert every `"?"` to a real missing value, using [`.replace()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html):


In [ ]:
adult_df = adult_df.replace('?', pd.NA)

adult_df.isna().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

Now `.isna()` tells the truth: **`workclass`**, **`occupation`**, and **`native.country`** all have real gaps. Everything downstream — imputation, standardization, the eventual pipeline — depends on catching this *first*.


## 3. Deduplication

Next: are any rows exact repeats of another row? [`.duplicated()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) flags them.


In [ ]:
print("Number of duplicate rows:", adult_df.duplicated().sum())

Number of duplicate rows: 24


Let's actually look at a couple of them — `keep=False` marks *every* copy of a duplicated row, not just the extras, so we can see the pairs side by side:


In [ ]:
adult_df.loc[adult_df.duplicated(keep=False)].head(6)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
6227,90,Private,52386,Some-college,10,Never-married,Other-service,Not-in-family,Asian-Pac-Islander,Male,0,0,35,United-States,<=50K
7615,19,Private,251579,Some-college,10,Never-married,Other-service,Own-child,White,Male,0,0,14,United-States,<=50K
7978,25,Private,308144,Bachelors,13,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,Mexico,<=50K
8356,21,Private,250051,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,10,United-States,<=50K
8453,25,Private,308144,Bachelors,13,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,Mexico,<=50K
8500,38,Private,207202,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,48,United-States,>50K


These are exact, full-row repeats — not just similar people, but the *same* record appearing twice. [`.drop_duplicates()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) keeps the first occurrence of each and removes the rest.


In [ ]:
adult_df = adult_df.drop_duplicates()

print("Shape after deduplication:", adult_df.shape)

Shape after deduplication: (32537, 15)


## 4. Imputation

Now that the sentinel values are real `NaN`s, we have a choice to make for each column — and it's not always the same choice.

Recall the Lesson 1.3 principle: **not every gap should be filled in.** For `workclass` and `occupation`, a missing value might genuinely mean something (e.g., someone who isn't employed) — imputing a fake job category would destroy that signal. For `native.country`, though, the data is heavily skewed toward one value, and the missing rate is low — a reasonable case for actual imputation.


In [ ]:
adult_df['native.country'].value_counts().head()

native.country
United-States    29153
Mexico             639
Philippines        198
Germany            137
Canada             121
Name: count, dtype: int64

The overwhelming majority of records are `United-States`. Filling the gaps with the [`.mode()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html) — the single most common value — using [`.fillna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) is a defensible choice here.


In [ ]:
most_common_country = adult_df['native.country'].mode()[0]
adult_df['native.country'] = adult_df['native.country'].fillna(most_common_country)

print("Remaining missing native.country:", adult_df['native.country'].isna().sum())

Remaining missing native.country: 0


For `workclass` and `occupation`, we'll follow the Lesson 1.3 principle instead: keep the gap as an explicit, honest category rather than guessing.


In [ ]:
adult_df['workclass'] = adult_df['workclass'].fillna('Unknown')
adult_df['occupation'] = adult_df['occupation'].fillna('Unknown')

print("Remaining missing values:")
print(adult_df[['workclass', 'occupation', 'native.country']].isna().sum())

Remaining missing values:
workclass         0
occupation        0
native.country    0
dtype: int64


## 5. Standardization (Conditional Mapping)

`marital.status` has seven categories, but several of them are really the same underlying idea spelled differently:


In [ ]:
adult_df['marital.status'].value_counts()

marital.status
Married-civ-spouse       14970
Never-married            10667
Divorced                  4441
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

**Conditional mapping** collapses many-to-one: a dictionary of old-value → new-value, applied with `.replace()`. Here, we'll consolidate down to two clear groups.


In [ ]:
marital_status_map = {
    'Married-civ-spouse': 'Married',
    'Married-AF-spouse': 'Married',
    'Married-spouse-absent': 'Married',
    'Never-married': 'Not-Married',
    'Divorced': 'Not-Married',
    'Separated': 'Not-Married',
    'Widowed': 'Not-Married',
}

adult_df['marital.status'] = adult_df['marital.status'].replace(marital_status_map)

adult_df['marital.status'].value_counts()

marital.status
Not-Married    17126
Married        15411
Name: count, dtype: int64

## 6. Fuzzy Matching

Real as this dataset is, it doesn't happen to contain any typos — every category is spelled consistently. To actually practice the technique, we'll **inject controlled, randomized typos** into a copy of the `education` column, then fix them back. This is clearly a synthetic step for practice purposes — everything before this section worked on the dataset's real, original messiness.

The function below picks a random subset of values and applies one small corruption each: swapping two adjacent letters, dropping a letter, or duplicating one. A fixed random seed means everyone gets the *same* "random" typos, so results are reproducible.


In [ ]:
import random

def introduce_typos(values, fraction=0.03, seed=42):
    # Return a copy of a string Series with a random fraction of values typo'd.
    rng = random.Random(seed)
    values = list(values)
    n_to_corrupt = int(len(values) * fraction)
    indices_to_corrupt = rng.sample(range(len(values)), n_to_corrupt)

    for i in indices_to_corrupt:
        word = values[i]
        if len(word) < 3:
            continue
        position = rng.randrange(len(word) - 1)
        typo_type = rng.choice(['swap', 'drop', 'duplicate'])

        if typo_type == 'swap':
            chars = list(word)
            chars[position], chars[position + 1] = chars[position + 1], chars[position]
            word = ''.join(chars)
        elif typo_type == 'drop':
            word = word[:position] + word[position + 1:]
        else:
            word = word[:position] + word[position] + word[position:]

        values[i] = word

    return pd.Series(values, index=range(len(values)))

In [ ]:
canonical_education = adult_df['education'].unique().tolist()

education_with_typos = introduce_typos(adult_df['education'].reset_index(drop=True), fraction=0.03, seed=42)

changed_mask = adult_df['education'].reset_index(drop=True) != education_with_typos
print("Number of values corrupted:", changed_mask.sum())

comparison = pd.DataFrame({
    'original': adult_df['education'].reset_index(drop=True)[changed_mask],
    'corrupted': education_with_typos[changed_mask]
})
comparison.head(10)

Number of values corrupted: 966


,original,corrupted
13,Masters,aMsters
18,Assoc-acdm,Assoc-cdm
55,Bachelors,Bacheelors
70,Bachelors,Bachlors
106,Prof-school,Prof-schoool
116,Bachelors,Baachelors
181,HS-grad,HS-gad
193,Prof-school,Pro-fschool
212,7th-8th,7th-t8h
235,Some-college,Some-collee


Now let's fix them. Python's built-in [`difflib.get_close_matches()`](https://docs.python.org/3/library/difflib.html#difflib.get_close_matches) compares a string against a list of known-good values and returns the closest match — no extra package needed.


In [ ]:
import difflib

def fix_typo(value, canonical_list, cutoff=0.7):
    # Match a possibly-corrupted value to its closest canonical value.
    if value in canonical_list:
        return value
    matches = difflib.get_close_matches(value, canonical_list, n=1, cutoff=cutoff)
    return matches[0] if matches else value

education_fixed = education_with_typos.apply(lambda v: fix_typo(v, canonical_education))

still_wrong = (education_fixed.values != adult_df['education'].reset_index(drop=True).values).sum()
print("Values still wrong after fuzzy-match fix:", still_wrong, "out of", changed_mask.sum(), "corrupted")

Values still wrong after fuzzy-match fix: 42 out of 966 corrupted


Not perfect — and that's worth sitting with. Let's see what fuzzy matching couldn't recover:


In [ ]:
still_wrong_mask = education_fixed.values != adult_df['education'].reset_index(drop=True).values
pd.DataFrame({
    'original': adult_df['education'].reset_index(drop=True)[still_wrong_mask],
    'corrupted': education_with_typos[still_wrong_mask],
    'fuzzy_fix': education_fixed[still_wrong_mask]
}).head(10)

,original,corrupted,fuzzy_fix
2778,11th,1th,12th
5064,10th,01th,12th
6089,10th,01th,12th
6883,11th,1th,12th
7002,10th,1th,12th
8024,11th,1th,12th
8101,10th,1t0h,12th
8698,9th,t9h,t9h
9086,11th,1t1h,12th
9125,10th,110th,11th


The failures cluster around **`9th`, `10th`, `11th`, `12th`** — short labels that are only one or two characters apart from each other to begin with. A typo in `"11th"` can land closer to `"12th"` than to the correct answer, purely by string similarity. This is a genuine limitation of fuzzy matching: it works well on longer, more distinctive strings, and gets shaky on short, similar-looking categories — which is exactly why fuzzy-matched output still deserves a human spot-check, not blind trust.


## 7. Functional Cleaning

Six manual steps is a lot to repeat by hand every time new data comes in. Let's wrap each technique into its own function, then chain them into a single pipeline.


In [ ]:
def convert_sentinels_to_missing(dataframe, sentinel_value='?'):
    # Replace a known sentinel value with a real missing value across the whole table.
    return dataframe.replace(sentinel_value, pd.NA)


def deduplicate(dataframe):
    # Drop exact duplicate rows, keeping the first occurrence.
    return dataframe.drop_duplicates()


def impute_missing(dataframe, mode_impute_columns, unknown_category_columns):
    # Fill missing values: mode for skewed/low-missing columns, an explicit
    # 'Unknown' category for columns where missingness may be meaningful.
    dataframe = dataframe.copy()
    for column in mode_impute_columns:
        most_common = dataframe[column].mode()[0]
        dataframe[column] = dataframe[column].fillna(most_common)
    for column in unknown_category_columns:
        dataframe[column] = dataframe[column].fillna('Unknown')
    return dataframe


def standardize_column(dataframe, column, mapping):
    # Apply a conditional mapping to consolidate category labels in one column.
    dataframe = dataframe.copy()
    dataframe[column] = dataframe[column].replace(mapping)
    return dataframe

Now the pipeline function — it calls each step in order, so cleaning new data becomes one function call instead of six manual ones.


In [ ]:
def clean_adult_data(dataframe):
    # Full cleaning pipeline for the Adult / Census Income dataset.
    dataframe = convert_sentinels_to_missing(dataframe, sentinel_value='?')
    dataframe = deduplicate(dataframe)
    dataframe = impute_missing(
        dataframe,
        mode_impute_columns=['native.country'],
        unknown_category_columns=['workclass', 'occupation']
    )
    dataframe = standardize_column(dataframe, 'marital.status', marital_status_map)
    return dataframe

Let's prove it works by running the pipeline on a **fresh copy of the original raw data** and comparing the result to what we built by hand, step by step.


In [ ]:
adult_raw = pd.read_csv(adult_url)
adult_pipeline_result = clean_adult_data(adult_raw)

print("Pipeline result shape:", adult_pipeline_result.shape)
print("Manual result shape:  ", adult_df.shape)
print("Remaining missing values (pipeline):")
print(adult_pipeline_result[['workclass', 'occupation', 'native.country']].isna().sum())

Pipeline result shape: (32537, 15)
Manual result shape:   (32537, 15)
Remaining missing values (pipeline):
workclass         0
occupation        0
native.country    0
dtype: int64


## 8. Guided Practice

One short exercise — fill in the `# TODO` line(s) and run the cell.


**Exercise.** The `race` column also uses `Amer-Indian-Eskimo` as a category label — outdated and inconsistent with how the other categories are named. Using the same conditional-mapping pattern as Section 5, write a mapping dictionary that renames `'Amer-Indian-Eskimo'` to `'American-Indian/Alaska-Native'`, leaving every other race category unchanged, and apply it with `.replace()`.

*Hint: `.replace()` with a dictionary only changes the keys you list — every other value passes through unchanged.*


In [ ]:
# TODO: build a mapping dictionary for the 'race' column and apply it with .replace()


## 9. Independent Activity

Back to **`nycflights13`** — the dataset from Notebook 01-01's Independent Activity. Its messiness looks different from the Adult dataset's: no sentinel values or typos here, just real, honest `NaN`s left behind by cancelled flights.

Your task:

1. Load `flights.csv` from the URL below into a DataFrame called `flights_df`.
2. Check `.isna().sum()` — which columns have missing values, and roughly how many?
3. Check `.duplicated().sum()` — are there any exact duplicate rows?
4. Using `.fillna()`, impute the missing `dep_delay` and `arr_delay` values with each column's **median** (not mean — delay data is skewed by a small number of very large delays, so the median is more representative). Store the result in `flights_cleaned`.
5. Confirm `flights_cleaned[['dep_delay', 'arr_delay']].isna().sum()` is now zero.

**Expected output:** a cleaned DataFrame where the delay columns have no missing values left — though `dep_time` and `arr_time` will still have gaps, since a cancelled flight genuinely has no departure or arrival time to impute.


In [ ]:
flights_url = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/flights/flights.csv"

# TODO: load flights_url into a DataFrame called flights_df


# TODO: check .isna().sum()


# TODO: check .duplicated().sum()


# TODO: impute missing dep_delay and arr_delay with the median of each column
#       -> store the result as flights_cleaned


# TODO: confirm dep_delay and arr_delay have no missing values left


## 10. Exporting the Cleaned Dataset

Cleaning a dataset is wasted effort if the result disappears when the notebook closes. [`.to_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) writes a DataFrame back out to a CSV file — `index=False` skips writing the row-number index as its own column, which is usually what you want for a clean export.


In [ ]:
adult_pipeline_result.to_csv('adult_cleaned.csv', index=False)

print("Saved. Rows:", adult_pipeline_result.shape[0], "| Columns:", adult_pipeline_result.shape[1])

Saved. Rows: 32537 | Columns: 15


In Colab, this file is written to the session's temporary storage — open the **folder icon** in the left sidebar to find and download `adult_cleaned.csv` directly. It won't persist once the runtime disconnects, so download it (or save it to Google Drive) if you need it later.


## 11. Wrap-Up

- A column can report **zero missing values** and still be hiding a sentinel — `.isna()` only catches what pandas already recognizes as missing
- **Deduplication**, **imputation**, and **standardization** each answer a different question: are these the same record twice? should this gap be filled or left explicit? are these different spellings of the same category?
- Imputation is a judgment call, not a default — sometimes the right move is to *not* fill a gap, and keep it as its own honest category instead
- **Fuzzy matching** recovers most typos, but isn't perfect — short, similar-looking categories are where it struggles most
- Wrapping each technique into a function, then chaining them into one pipeline, turns a six-step manual process into a single, repeatable function call

**Next up:** Distribution Analysis, Graph Reading & Transformation Techniques — where the cleaned data from this notebook becomes something we can actually visualize and model.
